# Amazon Customers Sentiment Analysis

**Goal:** Analyze Amazon product reviews, clean the text, classify sentiment (positive / negative / neutral),
find patterns in customer feedback and ratings, and turn them into insights for product improvement.

**How to use:** Put your reviews CSV next to this notebook, set `DATA_PATH` in the config cell, and run all cells top to bottom.
Works with Kaggle's *Amazon Fine Food Reviews* (`Reviews.csv`) and most other Amazon review datasets.

In [ ]:
%pip install -q vaderSentiment

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
LABELS = ["negative", "neutral", "positive"]
COLORS = {"negative": "#d1495b", "neutral": "#edae49", "positive": "#2e8b57"}

## Config
Set the file path. Leave the column names as `None` to auto-detect them (they are printed in the next step).
`SAMPLE_SIZE` caps the number of rows so the notebook stays fast on large datasets.

In [ ]:
DATA_PATH = "Reviews.csv"      # <-- change to your dataset file
TEXT_COL = None                # e.g. "Text"; None = auto-detect
RATING_COL = None              # e.g. "Score"; None = auto-detect
PRODUCT_COL = None             # e.g. "ProductId"; None = auto-detect (optional)
DATE_COL = None                # e.g. "Time"; None = auto-detect (optional)
SAMPLE_SIZE = 50000
RANDOM_STATE = 42

## Step 1: Load the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)

def find_col(candidates, given):
    if given:
        return given
    lowered = {c.lower(): c for c in df.columns}
    for cand in candidates:
        for low, orig in lowered.items():
            if cand in low:
                return orig
    return None

TEXT_COL = find_col(["reviews.text", "review_text", "reviewtext", "text", "review", "content"], TEXT_COL)
RATING_COL = find_col(["rating", "score", "stars", "overall"], RATING_COL)
PRODUCT_COL = find_col(["productid", "asin", "product_id", "product", "name"], PRODUCT_COL)
DATE_COL = find_col(["date", "time"], DATE_COL)

print("Text column   :", TEXT_COL)
print("Rating column :", RATING_COL)
print("Product column:", PRODUCT_COL)
print("Date column   :", DATE_COL)
assert TEXT_COL and RATING_COL, "Set TEXT_COL and RATING_COL manually in the config cell."
df.head()

## Step 2: Clean the data
Drop missing reviews, duplicates, and invalid ratings, then sample if the dataset is very large.

In [ ]:
n0 = len(df)
df = df.dropna(subset=[TEXT_COL, RATING_COL]).copy()
df[RATING_COL] = pd.to_numeric(df[RATING_COL], errors="coerce")
df = df.dropna(subset=[RATING_COL])
df = df[df[RATING_COL].between(1, 5)]
df[RATING_COL] = df[RATING_COL].round().astype(int)
df = df.drop_duplicates(subset=[TEXT_COL])
df = df[df[TEXT_COL].astype(str).str.strip().str.len() > 0]

if len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
df = df.reset_index(drop=True)
print(f"Rows: {n0} -> {len(df)} after cleaning")

## Step 3: Text preprocessing
Lowercase, remove HTML and URLs, strip non-letters, and drop stop-words.
Negation words (*not, no, never, ...*) are **kept** because they flip the meaning of a review.

In [ ]:
NEGATIONS = {"not", "no", "nor", "never", "cannot", "neither", "none", "nothing", "nobody", "nowhere"}
STOP = set(ENGLISH_STOP_WORDS) - NEGATIONS

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)            # html tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # urls
    text = re.sub(r"n't\b", " not", text)           # don't -> do not
    text = re.sub(r"[^a-z\s]", " ", text)           # keep letters only
    tokens = [t for t in text.split() if t not in STOP and len(t) > 2 or t in NEGATIONS]
    return " ".join(tokens)

df["raw_text"] = df[TEXT_COL].astype(str)
df["clean_text"] = df["raw_text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
df["word_count"] = df["raw_text"].str.split().str.len()

df[["raw_text", "clean_text"]].head(3)

## Step 4: Sentiment labels from ratings
Standard convention: **4-5 stars = positive, 3 stars = neutral, 1-2 stars = negative.**
These act as the ground truth to evaluate the other two approaches.

In [ ]:
def rating_to_sentiment(r):
    return "positive" if r >= 4 else ("neutral" if r == 3 else "negative")

df["sentiment"] = df[RATING_COL].apply(rating_to_sentiment)
print(df["sentiment"].value_counts())
print()
print(df["sentiment"].value_counts(normalize=True).round(3))

## Step 5: Lexicon-based sentiment (VADER)
VADER scores the **raw** text (capitals and punctuation like "!!!" carry signal).
Compound score: `>= 0.05` positive, `<= -0.05` negative, otherwise neutral.

In [ ]:
sia = SentimentIntensityAnalyzer()

def vader_label(text):
    c = sia.polarity_scores(text)["compound"]
    return c, ("positive" if c >= 0.05 else ("negative" if c <= -0.05 else "neutral"))

res = df["raw_text"].apply(vader_label)
df["vader_score"] = res.apply(lambda x: x[0])
df["vader_sentiment"] = res.apply(lambda x: x[1])

print("VADER vs rating-based label accuracy:", round(accuracy_score(df["sentiment"], df["vader_sentiment"]), 3))
print(classification_report(df["sentiment"], df["vader_sentiment"], labels=LABELS, zero_division=0))

## Step 6: Machine-learning classifier (TF-IDF + Logistic Regression)
Trains on the cleaned text to predict sentiment. `class_weight="balanced"` handles the usual
imbalance (most Amazon reviews are positive).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"], test_size=0.2, random_state=RANDOM_STATE, stratify=df["sentiment"]
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
Xtr = tfidf.fit_transform(X_train)
Xte = tfidf.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(Xtr, y_train)
pred = model.predict(Xte)

print("Test accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, labels=LABELS, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=LABELS)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix (TF-IDF + LogReg)")
plt.tight_layout(); plt.show()

## Step 7: Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Sentiment distribution
counts = df["sentiment"].value_counts().reindex(LABELS)
axes[0].bar(counts.index, counts.values, color=[COLORS[l] for l in counts.index])
axes[0].set_title("Sentiment distribution")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v / counts.sum():.0%}", ha="center", va="bottom")

# Rating distribution
rc = df[RATING_COL].value_counts().sort_index()
axes[1].bar(rc.index.astype(str), rc.values, color="#4c78a8")
axes[1].set_title("Rating distribution"); axes[1].set_xlabel("Stars")

# Review length by sentiment
plot_df = df[df["word_count"] <= df["word_count"].quantile(0.95)]
sns.boxplot(data=plot_df, x="sentiment", y="word_count", order=LABELS, palette=COLORS, ax=axes[2])
axes[2].set_title("Review length by sentiment (words)")

plt.tight_layout(); plt.show()

In [ ]:
# Sentiment trend over time (only if a date column exists)
if DATE_COL:
    d = df[DATE_COL]
    if pd.api.types.is_numeric_dtype(d):
        unit = "s" if d.median() < 1e11 else "ms"
        df["date"] = pd.to_datetime(d, unit=unit, errors="coerce")
    else:
        df["date"] = pd.to_datetime(d, errors="coerce")
    t = df.dropna(subset=["date"]).copy()
    if len(t):
        span_days = (t["date"].max() - t["date"].min()).days
        t["period"] = t["date"].dt.to_period("Q" if span_days > 730 else "M").dt.to_timestamp()
        trend = t.groupby(["period", "sentiment"]).size().unstack(fill_value=0).reindex(columns=LABELS, fill_value=0)
        share = trend.div(trend.sum(axis=1), axis=0) * 100
        ax = share.plot(color=[COLORS[c] for c in share.columns], figsize=(11, 4), marker="o")
        ax.set_ylabel("% of reviews"); ax.set_title("Sentiment share over time")
        plt.tight_layout(); plt.show()
        avg = t.groupby("period")[RATING_COL].mean()
        avg.plot(figsize=(11, 3), color="#4c78a8", marker="o", title="Average rating over time")
        plt.tight_layout(); plt.show()
else:
    print("No date column found, skipping the time trend.")

## Step 8: Patterns in customer feedback
The model's coefficients tell us which words push a review towards positive or negative.

In [ ]:
terms = np.array(tfidf.get_feature_names_out())
classes = list(model.classes_)
coef = pd.DataFrame(model.coef_.T, index=terms, columns=classes)

top_pos = coef["positive"].sort_values(ascending=False).head(15)
top_neg = coef["negative"].sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
top_pos[::-1].plot.barh(ax=axes[0], color=COLORS["positive"], title="Words / phrases signalling POSITIVE")
top_neg[::-1].plot.barh(ax=axes[1], color=COLORS["negative"], title="Words / phrases signalling NEGATIVE")
plt.tight_layout(); plt.show()

In [ ]:
# Where do rating and text disagree? (e.g. 5 stars but a negative-sounding review)
disagree = df[(df["sentiment"] == "positive") & (df["vader_sentiment"] == "negative")]
print(f"{len(disagree)} reviews are 4-5 stars but VADER reads them as negative. Examples:")
for t in disagree["raw_text"].head(3):
    print("-", t[:200])

# Products with the most negative feedback (only products with enough reviews)
if PRODUCT_COL:
    g = df.groupby(PRODUCT_COL).agg(reviews=("sentiment", "size"),
                                    negative_share=("sentiment", lambda s: (s == "negative").mean()),
                                    avg_rating=(RATING_COL, "mean"))
    worst = g[g["reviews"] >= max(5, g["reviews"].quantile(0.9))].sort_values("negative_share", ascending=False).head(10)
    display(worst.round(2))

## Step 9: Insights and recommendations

In [ ]:
pos_share = (df["sentiment"] == "positive").mean()
neg_share = (df["sentiment"] == "negative").mean()
neu_share = (df["sentiment"] == "neutral").mean()
len_by = df.groupby("sentiment")["word_count"].mean().round(1)

print(f"Reviews analysed       : {len(df):,}")
print(f"Positive / Neutral / Negative: {pos_share:.1%} / {neu_share:.1%} / {neg_share:.1%}")
print(f"Average rating         : {df[RATING_COL].mean():.2f}")
print(f"Avg review length (words): {len_by.to_dict()}")
print(f"ML model accuracy      : {accuracy_score(y_test, pred):.1%}")
print(f"VADER accuracy         : {accuracy_score(df['sentiment'], df['vader_sentiment']):.1%}")
print("Top negative signals   :", ", ".join(top_neg.index[:8]))
print("Top positive signals   :", ", ".join(top_pos.index[:8]))

### Write your conclusions here (use the numbers printed above)

- **Overall satisfaction:** share of positive vs negative reviews and the average rating.
- **What customers love:** themes from the top positive words (quality, taste, value, ...).
- **What drives complaints:** themes from the top negative words (price, packaging, damaged, stale, ...).
- **Rating vs text mismatch:** reviews where the stars and the written text disagree (they often hide specific issues).
- **Trend:** is sentiment improving or declining over time?
- **Recommendations:** 2-3 concrete actions, e.g. fix the most common complaint, protect what customers praise, monitor the worst products.

In [ ]:
out_cols = [c for c in [PRODUCT_COL, RATING_COL, "raw_text", "clean_text", "sentiment", "vader_score", "vader_sentiment"] if c]
df[out_cols].to_csv("amazon_reviews_with_sentiment.csv", index=False)
print("Saved amazon_reviews_with_sentiment.csv")